In [40]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [41]:
BASE_DIR = os.path.dirname(os.getcwd())
BASE_DIR

'c:\\Users\\jeanl\\OneDrive\\Desktop\\IMPORTANTES\\players-clustering'

Como os dados que extraímos estão separados entre diversas planilhas diferentes, precisamos de uma base central com as estatísticas mais interessantes. Vamos então criar essa base a partir das planilhas originais, normalizando algumas estatísticas para stat/90min que é a duração de uma partida de futebol, ou apenas mantendo o valor original em alguns casos. 

In [42]:
players_stats_path = Path('data/01_raw/players/all_stats_players.xlsx')
full_players_stats_path = os.path.join(BASE_DIR, players_stats_path)

full_players_stats_path

'c:\\Users\\jeanl\\OneDrive\\Desktop\\IMPORTANTES\\players-clustering\\data\\01_raw\\players\\all_stats_players.xlsx'

Vamos ler todos as abas da planilha de jogadores e criar ID's combinando nome do jogador + time + idade[anos-dias] para facilitar futuros merges.  

In [43]:
sheet_names=['Defense-Players', 
            'Gca-Players', 
            'Keeper-Adv-Players', 
            'Keeper-Players', 
            'Misc-Players', 
            'Passing-Players', 
            'Playing-Time-Players', 
            'Possession-Players', 
            'Shooting-Players', 
            'Standard-Players',
            'Passing-Types-Players']

dfs = {}

for sheet in sheet_names:
    df = pd.read_excel(full_players_stats_path, sheet_name=sheet)
    
    if 'player' in df.columns:
        df['player_id'] = df['player'] + '_' + df['team'] + '_' + df['age'].astype(str)
    else:
        print(f"Column 'player' not found in sheet: {sheet}")

    dfs[sheet] = df

In [44]:
df_defense = dfs['Defense-Players']
df_gca = dfs['Gca-Players']
df_keeper_adv = dfs['Keeper-Adv-Players']
df_keeper = dfs['Keeper-Players']
df_misc = dfs['Misc-Players']
df_passing = dfs['Passing-Players']
df_passing_types = dfs['Passing-Types-Players']
df_playing_time = dfs['Playing-Time-Players']
df_possession = dfs['Possession-Players']
df_shooting = dfs['Shooting-Players']
df_standard = dfs['Standard-Players']

Vamos seguir um padrão de colunas principais a se manter em todos os df's como sendo: *player_id*, *player*, *nationality*, *position*, *team*, *age*, *birth_year*

Da base de tempo de jogo vamos aproveitar a quantidade de jogos, minutos, minutos por jogo, proporção de minutos e jogos iniciados. Além de criar um novo dado para calcular a proporção de jogos iniciados com relação ao total de jogos. 

In [45]:
df_playing_time = df_playing_time[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'games', 'minutes', 'minutes_per_game', 'minutes_pct', 'games_starts']]

df_playing_time['games_starts_pct'] = df_playing_time['games_starts'] / df_playing_time['games']
df_playing_time['games_starts_pct'] = df_playing_time['games_starts_pct'].fillna(0).round(2)

df_playing_time = df_playing_time.dropna(subset=['minutes', 'minutes_pct', 'minutes_per_game'])

df_playing_time.head()

C:\Users\jeanl\AppData\Local\Temp\ipykernel_5220\2382637101.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_playing_time['games_starts_pct'] = df_playing_time['games_starts'] / df_playing_time['games']
C:\Users\jeanl\AppData\Local\Temp\ipykernel_5220\2382637101.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_playing_time['games_starts_pct'] = df_playing_time['games_starts_pct'].fillna(0).round(2)


,player_id,player,nationality,position,team,age,birth_year,games,minutes,minutes_per_game,minutes_pct,games_starts,games_starts_pct
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004.0,6,495.0,83.0,50.0,5,0.83
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999.0,7,299.0,43.0,30.2,2,0.29
3,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000.0,9,207.0,23.0,20.9,0,0.00
4,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000.0,10,888.0,89.0,89.7,10,1.00
5,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002.0,9,330.0,37.0,33.3,4,0.44


Observamos que a base de estatísticas "tradicionais" pode ser complementada à base de chutes para criar um **df_ofense**, que seria uma base com estatísticas ofensivas dos jogadores. Gols, chance de gols, chutes e derivados. 

In [46]:
new_cols_df_standard = [col for col in df_standard.columns if col not in df_shooting.columns]
cols_merge_df_standard = ['player_id', 'player'] + new_cols_df_standard

df_ofense = pd.merge(df_shooting, df_standard[cols_merge_df_standard], on=['player_id', 'player'], how='left')

Criando a diferença entre chance de gols e gols por 90 minutos.

In [47]:
# df_ofense.head()

df_ofense['xg_net_per90'] = df_ofense['goals_per90'] - df_ofense['xg_per90']

print(df_ofense.columns)

Index(['player', 'nationality', 'position', 'team', 'age', 'birth_year',
       'minutes_90s', 'goals', 'shots', 'shots_on_target',
       'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90',
       'goals_per_shot', 'goals_per_shot_on_target', 'average_shot_distance',
       'shots_free_kicks', 'pens_made', 'pens_att', 'xg', 'npxg',
       'npxg_per_shot', 'xg_net', 'npxg_net', 'matches', 'player_id', 'games',
       'games_starts', 'minutes', 'assists', 'goals_assists', 'goals_pens',
       'cards_yellow', 'cards_red', 'xg_assist', 'npxg_xg_assist',
       'progressive_carries', 'progressive_passes',
       'progressive_passes_received', 'goals_per90', 'assists_per90',
       'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90',
       'xg_per90', 'xg_assist_per90', 'xg_xg_assist_per90', 'npxg_per90',
       'npxg_xg_assist_per90', 'xg_net_per90'],
      dtype='object')


In [48]:
df_ofense = df_ofense[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'goals_per90', 'goals_pens_per90', 'shots_per90',
                       'shots_on_target_per90', 'shots_on_target_pct', 'goals_per_shot', 'goals_per_shot_on_target', 'xg_per90', 'npxg_per90',
                       'xg_net_per90', 'average_shot_distance']]

# df_ofense.fillna(0, inplace=True)
df_ofense.head()

,player_id,player,nationality,position,team,age,birth_year,goals_per90,goals_pens_per90,shots_per90,shots_on_target_per90,shots_on_target_pct,goals_per_shot,goals_per_shot_on_target,xg_per90,npxg_per90,xg_net_per90,average_shot_distance
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,0.0,0.0,0.18,0.00,0.0,0.00,NaN,0.02,0.02,-0.02,8.7
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,0.0,0.0,0.30,0.00,0.0,0.00,NaN,0.01,0.01,-0.01,22.1
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,0.0,0.0,1.30,0.87,66.7,0.00,0.0,0.06,0.06,-0.06,19.2
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,0.1,0.1,0.91,0.20,22.2,0.11,0.5,0.03,0.03,0.07,30.9
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,0.0,0.0,0.82,0.55,66.7,0.00,0.0,0.14,0.14,-0.14,17.9


Da mesma forma que o **df_standard** complementava o **df_shooting**, ele pode complementar o **df_passing**, então vamos criar um **df_passing_final** juntando dados relevantes entre essas bases. 

In [49]:
df_standard['assists_menos_xg_assist_per90'] = df_standard['assists_per90'] - df_standard['xg_assist_per90']

new_cols_df_passingt = [col for col in df_passing_types.columns if col not in df_passing.columns]
cols_merge_df_passingt = ['player_id', 'player'] + new_cols_df_passingt

df_passing_final = pd.merge(df_passing, df_standard[['player_id', 'player', 'assists_per90', 'xg_assist_per90', 'assists_menos_xg_assist_per90']], on=['player_id', 'player'], how='left')
df_passing_final = pd.merge(df_passing_final, df_passing_types[cols_merge_df_passingt], on=['player_id', 'player'], how='left')

PS: no processo também criamos a coluna que calcula diferença entre a chance de dar uma assistência e o total de assistências por 90 min.

In [50]:
df_passing_final.columns

Index(['player', 'nationality', 'position', 'team', 'age', 'birth_year',
       'minutes_90s', 'passes_completed', 'passes', 'passes_pct',
       'passes_total_distance', 'passes_progressive_distance',
       'passes_completed_short', 'passes_short', 'passes_pct_short',
       'passes_completed_medium', 'passes_medium', 'passes_pct_medium',
       'passes_completed_long', 'passes_long', 'passes_pct_long', 'assists',
       'xg_assist', 'pass_xa', 'xg_assist_net', 'assisted_shots',
       'passes_into_final_third', 'passes_into_penalty_area',
       'crosses_into_penalty_area', 'progressive_passes', 'matches',
       'player_id', 'assists_per90', 'xg_assist_per90',
       'assists_menos_xg_assist_per90', 'passes_live', 'passes_dead',
       'passes_free_kicks', 'through_balls', 'passes_switches', 'crosses',
       'throw_ins', 'corner_kicks', 'corner_kicks_in', 'corner_kicks_out',
       'corner_kicks_straight', 'passes_offsides', 'passes_blocked'],
      dtype='object')

Vamos normalizar as variáveis para mostrar os dados por 90 minutos ou proporcionalmente ao total de ações. 

In [51]:
df_passing_final['key_passes_per90'] = (df_passing_final['assisted_shots'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['attempted_passes_per90'] = (df_passing_final['passes'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['progressives_passes_per90'] = (df_passing_final['progressive_passes'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['passes_into_final_third_per90'] = (df_passing_final['passes_into_final_third'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['passes_into_penalty_area_per90'] = (df_passing_final['passes_into_penalty_area'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['crosses_per90'] = (df_passing_final['crosses'] / df_passing_final['minutes_90s']).round(2)
df_passing_final['long_pass_rate'] = (df_passing_final['passes_long'] / df_passing_final['passes']).round(2)

df_passing_final = df_passing_final[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'assists_per90', 'xg_assist_per90', 'assists_menos_xg_assist_per90', 'key_passes_per90',
                                     'attempted_passes_per90', 'passes_pct', 'passes_pct_short', 'passes_pct_medium', 'passes_pct_long', 'progressives_passes_per90', 'passes_into_final_third_per90',
                                     'passes_into_penalty_area_per90', 'crosses_per90', 'long_pass_rate']]

# df_passing_final.fillna(0, inplace=True)
df_passing_final.head()

,player_id,player,nationality,position,team,age,birth_year,assists_per90,xg_assist_per90,assists_menos_xg_assist_per90,...,attempted_passes_per90,passes_pct,passes_pct_short,passes_pct_medium,passes_pct_long,progressives_passes_per90,passes_into_final_third_per90,passes_into_penalty_area_per90,crosses_per90,long_pass_rate
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,0.18,0.03,0.15,...,45.64,84.9,92.6,94.1,48.9,0.73,0.91,0.18,0.00,0.18
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,0.00,0.04,-0.04,...,67.88,84.8,95.0,81.6,56.3,6.06,5.45,0.30,0.61,0.07
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,0.00,0.09,-0.09,...,62.17,83.9,89.7,78.1,66.7,5.22,5.22,1.74,3.04,0.04
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,0.00,0.05,-0.05,...,57.17,77.9,89.2,81.8,57.8,4.95,4.24,0.81,2.73,0.16
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,0.00,0.02,-0.02,...,18.65,73.9,76.5,85.0,50.0,2.43,0.54,0.00,1.35,0.09


Vamos normalizar os dados do **df_possession** por 90 minutos para criar o **df_possession_final**.

In [52]:
df_possession.columns

Index(['player', 'nationality', 'position', 'team', 'age', 'birth_year',
       'minutes_90s', 'touches', 'touches_def_pen_area', 'touches_def_3rd',
       'touches_mid_3rd', 'touches_att_3rd', 'touches_att_pen_area',
       'touches_live_ball', 'take_ons', 'take_ons_won', 'take_ons_won_pct',
       'take_ons_tackled', 'take_ons_tackled_pct', 'carries',
       'carries_distance', 'carries_progressive_distance',
       'progressive_carries', 'carries_into_final_third',
       'carries_into_penalty_area', 'miscontrols', 'dispossessed',
       'passes_received', 'progressive_passes_received', 'matches',
       'player_id'],
      dtype='object')

In [53]:
df_possession_final = df_possession.copy()
df_possession_final['touches_per90'] = (df_possession_final['touches'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['touches_defensive_third_per90'] = (df_possession_final['touches_def_3rd'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['touches_middle_third_per90'] = (df_possession_final['touches_mid_3rd'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['touches_attacking_third_per90'] = (df_possession_final['touches_att_3rd'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['touches_penalty_area_per90'] = (df_possession_final['touches_att_pen_area'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['take_ons_won_per90'] = (df_possession_final['take_ons_won'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['progressive_carries_per90'] = (df_possession_final['progressive_carries'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['passes_received_per90'] = (df_possession_final['passes_received'] / df_possession_final['minutes_90s']).round(2)
df_possession_final['dispossessions_per90'] = (df_possession_final['dispossessed'] / df_possession_final['minutes_90s']).round(2)

df_possession_final = df_possession_final[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'touches_per90', 'touches_defensive_third_per90', 'touches_middle_third_per90',
                                           'touches_attacking_third_per90', 'touches_penalty_area_per90', 'take_ons_won_per90', 'take_ons_won_pct', 'progressive_carries_per90', 
                                           'passes_received_per90', 'dispossessions_per90']]
# df_possession_final.fillna(0, inplace=True)

df_possession_final.head()

,player_id,player,nationality,position,team,age,birth_year,touches_per90,touches_defensive_third_per90,touches_middle_third_per90,touches_attacking_third_per90,touches_penalty_area_per90,take_ons_won_per90,take_ons_won_pct,progressive_carries_per90,passes_received_per90,dispossessions_per90
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,56.00,38.00,18.00,0.73,0.55,0.73,100.0,0.55,38.00,0.18
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,76.67,23.94,40.91,13.64,0.91,0.91,60.0,2.12,52.12,2.42
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,75.65,7.39,33.48,35.65,2.17,1.30,20.0,2.61,63.04,0.43
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,69.70,18.99,31.82,19.19,0.61,0.81,50.0,1.52,41.11,0.20
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,31.08,5.14,12.70,14.05,3.51,1.08,30.8,1.35,16.49,0.54


Analisamos o **df_defense** e **df_misc** e percebemos que são mais 2 df's que contém dados complementares e podem ser concatenados para criar o **df_defense_final**. 

In [54]:
print(df_defense.columns)

print(df_misc.columns)

Index(['player', 'nationality', 'position', 'team', 'age', 'birth_year',
       'minutes_90s', 'tackles', 'tackles_won', 'tackles_def_3rd',
       'tackles_mid_3rd', 'tackles_att_3rd', 'challenge_tackles', 'challenges',
       'challenge_tackles_pct', 'challenges_lost', 'blocks', 'blocked_shots',
       'blocked_passes', 'interceptions', 'tackles_interceptions',
       'clearances', 'errors', 'matches', 'player_id'],
      dtype='object')
Index(['player', 'nationality', 'position', 'team', 'age', 'birth_year',
       'minutes_90s', 'cards_yellow', 'cards_red', 'cards_yellow_red', 'fouls',
       'fouled', 'offsides', 'crosses', 'interceptions', 'tackles_won',
       'pens_won', 'pens_conceded', 'own_goals', 'ball_recoveries',
       'aerials_won', 'aerials_lost', 'aerials_won_pct', 'matches',
       'player_id'],
      dtype='object')


In [55]:
df_defense_final = pd.merge(df_defense, df_misc[['player_id', 'player', 'aerials_won', 'aerials_won_pct']], on=['player', 'player_id'], how='left')
df_defense_final.columns
print(df_defense_final.head())

            player nationality position           team     age  birth_year  \
0            Abner      br BRA       ZG      Juventude  21-044        2004   
1  Nicolás Acevedo      uy URU    LT.ZG          Bahia  26-049        1999   
2            Adson      br BRA    AT.LT  Vasco da Gama  24-239        2000   
3   Braian Aguirre      ar ARG       ZG  Internacional  24-309        2000   
4   Carlos Alberto      br BRA    AT.LT   Sport Recife  23-049        2002   

   minutes_90s  tackles  tackles_won  tackles_def_3rd  ...  blocked_shots  \
0          5.5      5.0            2              4.0  ...            7.0   
1          3.3      8.0            5              2.0  ...            1.0   
2          2.3      6.0            4              4.0  ...            0.0   
3          9.9     27.0           13             10.0  ...            2.0   
4          3.7      4.0            4              2.0  ...            0.0   

   blocked_passes  interceptions  tackles_interceptions  clearances 

In [56]:
df_defense_final['tackles_per90'] = (df_defense_final['tackles'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['tackles_won_per90'] = (df_defense_final['tackles_won'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['tackles_won_pct'] = (df_defense_final['tackles_won'] / df_defense_final['tackles']).round(2)
df_defense_final['interceptions_per90'] = (df_defense_final['interceptions'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['blocks_per90'] = (df_defense_final['blocks'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['clearances_per90'] = (df_defense_final['clearances'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['errors_per90'] = (df_defense_final['errors'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['defensive_actions'] = df_defense_final['tackles'] + df_defense_final['interceptions'] + df_defense_final['blocks'] + df_defense_final['clearances']
df_defense_final['defensive_actions_per90'] = (df_defense_final['defensive_actions'] / df_defense_final['minutes_90s']).round(2)
df_defense_final['aerials_won_per90'] = (df_defense_final['aerials_won'] / df_defense_final['minutes_90s']).round(2)

df_defense_final = df_defense_final[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'tackles_per90', 'tackles_won_per90', 'tackles_won_pct', 'interceptions_per90',
                                     'blocks_per90', 'clearances_per90', 'errors_per90', 'defensive_actions_per90', 'aerials_won_per90', 'aerials_won_pct']]

# df_defense_final.fillna(0, inplace=True)
df_defense_final.head()

,player_id,player,nationality,position,team,age,birth_year,tackles_per90,tackles_won_per90,tackles_won_pct,interceptions_per90,blocks_per90,clearances_per90,errors_per90,defensive_actions_per90,aerials_won_per90,aerials_won_pct
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,0.91,0.36,0.40,1.45,1.82,5.64,0.0,9.82,2.91,80.0
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,2.42,1.52,0.62,1.21,1.21,1.21,0.0,6.06,1.52,71.4
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,2.61,1.74,0.67,0.00,0.87,1.74,0.0,5.22,0.43,20.0
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,2.73,1.31,0.48,1.11,1.31,3.23,0.1,8.38,0.40,28.6
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,1.08,1.08,1.00,2.43,0.54,1.89,0.0,5.95,1.35,33.3


Nos próximos passos apenas normalizamos os **df_gca** e **df_misc** por 90 min ou proporção por total de ações e criamos o **df_gca_final** e **df_misc_final**

In [57]:
df_gca_final = df_gca.copy()
df_gca_final['sca_passes_live_per90'] = (df_gca_final['sca_passes_live'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['sca_passes_dead_per90'] = (df_gca_final['sca_passes_dead'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['sca_take_ons_per90'] = (df_gca_final['sca_take_ons'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['sca_shots_per90'] = (df_gca_final['sca_shots'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['sca_fouled_per90'] = (df_gca_final['sca_fouled'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['gca_passes_live_per90'] = (df_gca_final['gca_passes_live'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['gca_passes_dead_per90'] = (df_gca_final['gca_passes_dead'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['gca_take_ons_per90'] = (df_gca_final['gca_take_ons'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['gca_shots_per90'] = (df_gca_final['gca_shots'] / df_gca_final['minutes_90s']).round(2)
df_gca_final['gca_fouled_per90'] = (df_gca_final['gca_fouled'] / df_gca_final['minutes_90s']).round(2)

df_gca_final = df_gca_final[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'sca_per90', 'sca_passes_live_per90', 'sca_passes_dead_per90', 'sca_take_ons_per90',
                             'sca_shots_per90', 'sca_fouled_per90', 'gca_per90', 'gca_passes_live_per90', 'gca_passes_dead_per90', 'gca_take_ons_per90', 
                             'gca_shots_per90', 'gca_fouled_per90']]

# df_gca_final.fillna(0, inplace=True)
df_gca_final.head()

,player_id,player,nationality,position,team,age,birth_year,sca_per90,sca_passes_live_per90,sca_passes_dead_per90,sca_take_ons_per90,sca_shots_per90,sca_fouled_per90,gca_per90,gca_passes_live_per90,gca_passes_dead_per90,gca_take_ons_per90,gca_shots_per90,gca_fouled_per90
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,0.18,0.18,0.0,0.00,0.0,0.0,0.18,0.18,0.0,0.0,0.0,0.0
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,0.60,0.30,0.0,0.00,0.3,0.0,0.30,0.30,0.0,0.0,0.0,0.0
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,4.35,3.91,0.0,0.43,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,1.52,1.31,0.0,0.00,0.1,0.0,0.00,0.00,0.0,0.0,0.0,0.0
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,2.45,2.16,0.0,0.00,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0


In [58]:
df_misc_final = df_misc.copy()
df_misc_final['fouls_per90'] = (df_misc_final['fouls'] / df_misc_final['minutes_90s']).round(2)
df_misc_final['fouls_drawn_per90'] = (df_misc_final['fouled'] / df_misc_final['minutes_90s']).round(2)
df_misc_final['fouls_drawn_pct'] = np.where(df_misc_final['fouls'] != 0, (df_misc_final['fouled'] / df_misc_final['fouls']).round(2), 0)
df_misc_final['yellow_cards_per90'] = (df_misc_final['cards_yellow'] / df_misc_final['minutes_90s']).round(2)
df_misc_final['red_cards_per90'] = (df_misc_final['cards_red'] / df_misc_final['minutes_90s']).round(2)
df_misc_final['ball_recoveries_per90'] = (df_misc_final['ball_recoveries'] / df_misc_final['minutes_90s']).round(2)
df_misc_final['offsides_per90'] = (df_misc_final['offsides'] / df_misc_final['minutes_90s']).round(2)

df_misc_final = df_misc_final[['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year', 'fouls_per90', 'fouls_drawn_per90', 'fouls_drawn_pct', 'yellow_cards_per90',
                               'red_cards_per90', 'ball_recoveries_per90', 'offsides_per90']]

# df_misc_final.fillna(0, inplace=True)
df_misc_final.head()

,player_id,player,nationality,position,team,age,birth_year,fouls_per90,fouls_drawn_per90,fouls_drawn_pct,yellow_cards_per90,red_cards_per90,ball_recoveries_per90,offsides_per90
0,Abner_Juventude_21-044,Abner,br BRA,ZG,Juventude,21-044,2004,0.36,0.36,1.00,0.18,0.0,2.73,0.18
1,Nicolás Acevedo_Bahia_26-049,Nicolás Acevedo,uy URU,LT.ZG,Bahia,26-049,1999,0.91,0.91,1.00,0.30,0.0,6.67,0.00
2,Adson_Vasco da Gama_24-239,Adson,br BRA,AT.LT,Vasco da Gama,24-239,2000,0.43,1.30,3.00,0.00,0.0,3.04,0.00
3,Braian Aguirre_Internacional_24-309,Braian Aguirre,ar ARG,ZG,Internacional,24-309,2000,0.91,0.10,0.11,0.10,0.0,4.14,0.20
4,Carlos Alberto_Sport Recife_23-049,Carlos Alberto,br BRA,AT.LT,Sport Recife,23-049,2002,1.89,0.00,0.00,0.27,0.0,3.51,0.27


Por fim, agregamos os dados e exportamos para /data/02_intermediate onde serão feitos futuros tratamentos para melhorar a qualidade dos dados.

In [59]:
common_cols = ['player_id', 'player', 'nationality', 'position', 'team', 'age', 'birth_year']

df_final = pd.merge(df_playing_time, df_ofense, on=common_cols, how='left')
df_final = pd.merge(df_final, df_passing_final, on=common_cols, how='left')
df_final = pd.merge(df_final, df_possession_final, on=common_cols, how='left')
df_final = pd.merge(df_final, df_defense_final, on=common_cols, how='left')
df_final = pd.merge(df_final, df_gca_final, on=common_cols, how='left')
df_final = pd.merge(df_final, df_misc_final, on=common_cols, how='left')
# df_final.fillna(0, inplace=True)

output_path = Path('data/02_intermediate/initial_data.xlsx')
full_output_path = os.path.join(BASE_DIR, output_path)

df_final.to_excel(full_output_path, index=False)


Todo: estudar qual a melhor prática para identificar valores Nan para o modelo, os quais não podem ser preenchidos com 0. ex: em "goals_per_shot" o jogador fulano tem valor Nan, porque ele nem chegou a chutar. Porém jogador ciclano tem valor 0, porque chutou e não fez gol. Esses dois casos devem ser diferenciados, as possíveis soluções que analisei até agora foram:

1. Imputar -1 para jogadores que não chutaram a gol, visto que a base só tem valores positivos, não seria um problema. 

2. Criar uma coluna binária indicando se o jogador chutou ou não a gol (0 e 1)

Provavelmente a alternativa de engenharia de feature (2) é a mais interessante em termos de precisão do modelo. 

Isso segue para várias colunas da base a serem anotadas depois. 

Para o modelo é provável que a melhor alternativa seja usar Agrupamento com o objetivo de identificar quais jogadores tem o mesmo estilo de jogo estatisticamente falando, gerando uma visão para além das posições exclusivamente. 